### Fig3

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import xarray as xr

# ==============================================================================
# --- 1. 路徑與參數設定 ---
# ==============================================================================
csv_40folds_path = 'all_40folds_39years_PO_AO_PCs_1981_2020_regenerated.csv'
recruitment_csv_path = 'recruitment.csv'
sst_path = 'sst.mnmean1.nc'
chl_path = 'GlobColour_19970901-20230603_CHL1_daily.nc'
output_dir = 'Fig3_LOOCV_MethodA/'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# ==============================================================================
# --- 2. 讀取生活史環境資料函數 ---
# ==============================================================================
def process_env_data(file_path, lat_range, lon_range, months, var_name, start_year, end_year='2020'):
    ds = xr.open_dataset(file_path)
    lat_min, lat_max = min(lat_range), max(lat_range)
    lon_min, lon_max = min(lon_range), max(lon_range)
    try:
        subset = ds.sel(time=slice(start_year, end_year), lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))
        if len(subset.lat) == 0:
            subset = ds.sel(time=slice(start_year, end_year), lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    except:
        subset = ds.sel(time=slice(start_year, end_year), lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

    month_mask = subset['time.month'].isin(months)
    ds_mm = subset.where(month_mask, drop=True)
    yearly_mean = ds_mm.groupby('time.year').mean('time')
    return yearly_mean[var_name].mean(dim=['lat', 'lon']).to_series()

env_configs = {
    'sst46':   {'path': sst_path, 'lat': [30, 18], 'lon': [122, 130], 'mos': [4,5,6],    'var': 'sst',  'start': '1981'},
    'sst79':   {'path': sst_path, 'lat': [40, 35], 'lon': [131, 140], 'mos': [7,8,9],    'var': 'sst',  'start': '1981'},
    'sst1012': {'path': sst_path, 'lat': [38, 30], 'lon': [125, 136], 'mos': [10,11,12], 'var': 'sst',  'start': '1981'},
    'chl46':   {'path': chl_path, 'lat': [30, 18], 'lon': [122, 130], 'mos': [4,5,6],    'var': 'CHL',  'start': '1998'},
    'chl79':   {'path': chl_path, 'lat': [40, 35], 'lon': [131, 140], 'mos': [7,8,9],    'var': 'CHL',  'start': '1998'},
    'chl1012': {'path': chl_path, 'lat': [38, 30], 'lon': [125, 136], 'mos': [10,11,12], 'var': 'CHL',  'start': '1998'},
}

print("正在處理生活史環境變數資料...")
env_results = {name: process_env_data(cfg['path'], cfg['lat'], cfg['lon'], cfg['mos'], cfg['var'], cfg['start']) 
               for name, cfg in env_configs.items()}

# 讀入 40 折大表與 Recruitment
df_40folds = pd.read_csv(csv_40folds_path)
df_rec = pd.read_csv(recruitment_csv_path)
year_col = 'Year' if 'Year' in df_rec.columns else 'year' if 'year' in df_rec.columns else df_rec.columns[0]
df_rec = df_rec.set_index(year_col)
rec_col = [c for c in df_rec.columns if 'recruitment' in c.lower()][0]
df_rec['log_recruitment'] = np.log10(df_rec[rec_col] * 1000)

# ==============================================================================
# --- 3. 定義 7 張圖的個別配置 ---
# ==============================================================================
plot_items = [
    {'title': 'PO Mode 1',   'file_name': 'Fig3_PO_Mode_1',   'var': 'PO_PC1',         'is_loocv': True},
    {'title': 'PO Mode 2',   'file_name': 'Fig3_PO_Mode_2',   'var': 'PO_PC2',         'is_loocv': True},
    {'title': 'PO Mode 3',   'file_name': 'Fig3_PO_Mode_3',   'var': 'PO_PC3',         'is_loocv': True},
    {'title': 'AO Mode 1',   'file_name': 'Fig3_AO_Mode_1',   'var': 'AO_PC1',         'is_loocv': True},
    {'title': 'AO Mode 2',   'file_name': 'Fig3_AO_Mode_2',   'var': 'AO_PC2',         'is_loocv': True},
    {'title': 'AO Mode 3',   'file_name': 'Fig3_AO_Mode_3',   'var': 'AO_PC3',         'is_loocv': True},
    {'title': 'Recruitment', 'file_name': 'Fig3_Recruitment', 'var': 'log_recruitment', 'is_loocv': False}
]

seasons = [
    {'label': 'Apr - Jun', 'sst': 'sst46',   'chl': 'chl46'},
    {'label': 'Jul - Sep', 'sst': 'sst79',   'chl': 'chl79'},
    {'label': 'Oct - Dec', 'sst': 'sst1012', 'chl': 'chl1012'}
]

# ==============================================================================
# --- 4. 批次產出 7 張獨立圖表 ---
# ==============================================================================
for i, item in enumerate(plot_items):
    fig, ax = plt.subplots(figsize=(12, 6))
    
    y_labels = []
    r_sst_mean, r_sst_err, p_sst_sig = [], [], []
    r_chl_mean, r_chl_err, p_chl_sig = [], [], []
    
    var = item['var']
    
    for season in seasons:
        y_labels.append(season['label'])
        
        if item['is_loocv']:
            r_sst_folds, p_sst_folds = [], []
            r_chl_folds, p_chl_folds = [], []
            
            # 遍歷 40 折計算相關係數與個別 p-value
            for fold_id, group_fold in df_40folds.groupby('Fold'):
                s_pc = group_fold.set_index('Train_Year')[var]
                
                # SST 相關
                comb_sst = pd.concat([s_pc, env_results[season['sst']]], axis=1).dropna()
                if len(comb_sst) > 5:
                    r_val, p_val = stats.pearsonr(comb_sst.iloc[:, 0], comb_sst.iloc[:, 1])
                    r_sst_folds.append(r_val)
                    p_sst_folds.append(p_val)
                
                # CHL 相關
                comb_chl = pd.concat([s_pc, env_results[season['chl']]], axis=1).dropna()
                if len(comb_chl) > 5:
                    r_val, p_val = stats.pearsonr(comb_chl.iloc[:, 0], comb_chl.iloc[:, 1])
                    r_chl_folds.append(r_val)
                    p_chl_folds.append(p_val)
            
            # Mean & SD
            m_sst = np.mean(r_sst_folds) if len(r_sst_folds) > 0 else 0
            s_sst = np.std(r_sst_folds) if len(r_sst_folds) > 0 else 0
            m_chl = np.mean(r_chl_folds) if len(r_chl_folds) > 0 else 0
            s_chl = np.std(r_chl_folds) if len(r_chl_folds) > 0 else 0
            
            r_sst_mean.append(m_sst)
            r_sst_err.append(s_sst)
            r_chl_mean.append(m_chl)
            r_chl_err.append(s_chl)
            
            # 判斷 40 折的 p-value 是否全部小於 0.05
            threshold = 0.90
            is_sst_all_sig = (sum(p < 0.05 for p in p_sst_folds) / len(p_sst_folds)) >= threshold if len(p_sst_folds) > 0 else False
            is_chl_all_sig = (sum(p < 0.05 for p in p_chl_folds) / len(p_chl_folds)) >= threshold if len(p_chl_folds) > 0 else False
            
            p_sst_sig.append(is_sst_all_sig)
            p_chl_sig.append(is_chl_all_sig)
            
        else:
            # Recruitment 使用 40 年點估計
            s_rec = df_rec[var]
            
            comb_sst = pd.concat([s_rec, env_results[season['sst']]], axis=1).dropna()
            r_val_s, p_val_s = stats.pearsonr(comb_sst.iloc[:, 0], comb_sst.iloc[:, 1]) if len(comb_sst) > 5 else (0, 1.0)
            r_sst_mean.append(r_val_s)
            r_sst_err.append(0)
            p_sst_sig.append(p_val_s < 0.05)
            
            comb_chl = pd.concat([s_rec, env_results[season['chl']]], axis=1).dropna()
            r_val_c, p_val_c = stats.pearsonr(comb_chl.iloc[:, 0], comb_chl.iloc[:, 1]) if len(comb_chl) > 5 else (0, 1.0)
            r_chl_mean.append(r_val_c)
            r_chl_err.append(0)
            p_chl_sig.append(p_val_c < 0.05)

    y_coords = np.arange(len(y_labels))
    bar_width = 0.35

    # 針對 Recruitment (非 LOOCV)，將 error bar 參數完全設為 None，徹底避免繪製出黑線
    xerr_sst = r_sst_err if item['is_loocv'] else None
    xerr_chl = r_chl_err if item['is_loocv'] else None
    err_kw = {'elinewidth': 1.5, 'ecolor': 'black'} if item['is_loocv'] else {}
    cap_sz = 5 if item['is_loocv'] else 0

    # 繪製 SST (上方) 與 CHL (下方)
    bars_sst = ax.barh(y_coords - bar_width/2, r_sst_mean, bar_width, 
                       xerr=xerr_sst, capsize=cap_sz, error_kw=err_kw,
                       label='SST', color='royalblue', alpha=0.85)
    bars_chl = ax.barh(y_coords + bar_width/2, r_chl_mean, bar_width, 
                       xerr=xerr_chl, capsize=cap_sz, error_kw=err_kw,
                       label='CHL', color='darkorange', alpha=0.85)

    # 顯著性打星號
    for idx, (bar, sig, err) in enumerate(zip(bars_sst, p_sst_sig, r_sst_err)):
        if sig:
            offset = 0.04 if bar.get_width() >= 0 else -0.07
            x_pos = bar.get_width() + (err if bar.get_width() >= 0 else -err) + offset
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, '*', 
                    va='center', ha='center', fontsize=26, color='darkblue')

    for idx, (bar, sig, err) in enumerate(zip(bars_chl, p_chl_sig, r_chl_err)):
        if sig:
            offset = 0.04 if bar.get_width() >= 0 else -0.07
            x_pos = bar.get_width() + (err if bar.get_width() >= 0 else -err) + offset
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, '*', 
                    va='center', ha='center', fontsize=26, color='brown')

    # 各季節間的分隔水平線
    for y in range(len(y_labels) - 1):
        ax.axhline(y + 0.5, color='black', linewidth=0.8, alpha=0.4)

    ax.set_yticks(y_coords)
    ax.set_yticklabels(y_labels, fontsize=22)
    ax.invert_yaxis()

    # 圖表標題與標籤
    ax.set_title(f"({chr(97+i)}) {item['title']}", loc='left', fontsize=26, pad=18)
    ax.set_xlabel('Correlation Coefficient (r)', fontsize=22)
    ax.tick_params(axis='x', labelsize=20)
    ax.axvline(0, color='black', linewidth=1.5)
    ax.set_xlim(-1.1, 1.1)
    ax.legend(loc='upper right', fontsize=16, frameon=True)

    plt.tight_layout()
    
    # 存檔（PNG 與 SVG 皆儲存）
    save_path_png = os.path.join(output_dir, f"{item['file_name']}_1029_90%.png")
    save_path_svg = os.path.join(output_dir, f"{item['file_name']}_1029_90%.svg")
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(save_path_svg, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[{i+1}/7] 已成功儲存圖表：{save_path_png}")

### another area

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import xarray as xr

# ==============================================================================
# --- 1. 路徑與參數設定 ---
# ==============================================================================
mask_base_dir = (
    "Fig3/"
)
csv_40folds_path = (
    "all_40folds_39years_PO_AO_PCs_1981_2020.csv"
)
sst_path = "sst.mnmean1.nc"
chl_path = (
    "GlobColour_19970901-20230603_CHL1_daily.nc"
)
output_dir = (
    "Fig3_LOOCV_Masks/"
)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 4 個目標遮罩檔案設定 (對應月份、檔名及擷取之標題)
mask_configs = [
    {
        'file_name': 'mask_AMJ_hatch.r1440x721.EA.nc',
        'tag': 'AMJ_hatch',
        'months': [4, 5, 6],
    },
    {
        'file_name': 'mask_AMJ_lt131.r1440x721.EA.nc',
        'tag': 'AMJ_lt131',
        'months': [4, 5, 6],
    },
    {
        'file_name': 'mask_JAS_JpSea.r1440x721.EA.nc',
        'tag': 'JAS_JpSea',
        'months': [7, 8, 9],
    },
    {
        'file_name': 'mask_OND_boxes.r1440x721.EA.nc',
        'tag': 'OND_boxes',
        'months': [10, 11, 12],
    },
]

# 6 個 PC 標籤與順序
pc_list = [
    'PO_PC1',
    'PO_PC2',
    'PO_PC3',
    'AO_PC1',
    'AO_PC2',
    'AO_PC3',
]

# 設定過檢定比例門檻為 80% (0.8)
SIG_THRESHOLD = 0.65

# ==============================================================================
# --- 2. 函數：透過遮罩提取年平均環境時間序列 ---
# ==============================================================================
def process_masked_env_data(
    env_ds, var_name, mask_da, months, start_year, end_year='2020'
):
    """將遮罩最近鄰插值到環境網格，選取 mask == 1 後計算目標月份的區域年平均。"""
    interp_dict = {}
    if 'lat' in env_ds.dims and 'lat' in mask_da.dims:
        interp_dict['lat'] = env_ds['lat']
    elif 'latitude' in env_ds.dims and 'latitude' in mask_da.dims:
        interp_dict['latitude'] = env_ds['latitude']

    if 'lon' in env_ds.dims and 'lon' in mask_da.dims:
        interp_dict['lon'] = env_ds['lon']
    elif 'longitude' in env_ds.dims and 'longitude' in mask_da.dims:
        interp_dict['longitude'] = env_ds['longitude']

    mask_interp = mask_da.interp(interp_dict, method='nearest')
    binary_mask = mask_interp == 1

    subset = env_ds[var_name].sel(time=slice(start_year, end_year))
    month_mask = subset['time.month'].isin(months)
    subset_months = subset.where(month_mask, drop=True)

    masked_data = subset_months.where(binary_mask)

    spatial_dims = [dim for dim in ['lat', 'lon', 'latitude', 'longitude'] if dim in masked_data.dims]
    spatial_mean = masked_data.mean(dim=spatial_dims, skipna=True)
    yearly_mean = spatial_mean.groupby('time.year').mean('time', skipna=True)

    return yearly_mean.to_series()


# ==============================================================================
# --- 3. 載入資料 ---
# ==============================================================================
print('正在載入環境資料與 40 折大表...')
ds_sst = xr.open_dataset(sst_path)
ds_chl = xr.open_dataset(chl_path)
df_40folds = pd.read_csv(csv_40folds_path)

# ==============================================================================
# --- 4. 針對 4 個遮罩分別計算相關性並繪圖 ---
# ==============================================================================
for config in mask_configs:
    tag = config['tag']
    months = config['months']
    mask_path = os.path.join(mask_base_dir, config['file_name'])

    print(f'\n正在處理遮罩區域：{tag} (月份: {months})...')
    ds_mask = xr.open_dataset(mask_path)
    da_mask = ds_mask['mask']

    s_sst = process_masked_env_data(
        ds_sst, 'sst', da_mask, months, start_year='1981'
    )
    s_chl = process_masked_env_data(
        ds_chl, 'CHL', da_mask, months, start_year='1998'
    )

    r_sst_mean, r_sst_err, p_sst_sig = [], [], []
    r_chl_mean, r_chl_err, p_chl_sig = [], [], []

    # 計算 6 個 PC 的 40 折 LOOCV 相關性
    for var in pc_list:
        r_sst_folds, p_sst_folds = [], []
        r_chl_folds, p_chl_folds = [], []

        for fold_id, group_fold in df_40folds.groupby('Fold'):
            s_pc = group_fold.set_index('Train_Year')[var]

            # SST 相關與 p-value
            comb_sst = pd.concat([s_pc, s_sst], axis=1).dropna()
            if len(comb_sst) > 5:
                r_val, p_val = stats.pearsonr(comb_sst.iloc[:, 0], comb_sst.iloc[:, 1])
                r_sst_folds.append(r_val)
                p_sst_folds.append(p_val)

            # CHL 相關與 p-value
            comb_chl = pd.concat([s_pc, s_chl], axis=1).dropna()
            if len(comb_chl) > 5:
                r_val, p_val = stats.pearsonr(comb_chl.iloc[:, 0], comb_chl.iloc[:, 1])
                r_chl_folds.append(r_val)
                p_chl_folds.append(p_val)

        # SST 統計量 (Mean, SD, 80% 門檻檢定)
        m_sst = np.mean(r_sst_folds) if len(r_sst_folds) > 0 else 0
        s_sst_std = np.std(r_sst_folds) if len(r_sst_folds) > 0 else 0
        # 判定 p < 0.05 的折數比例是否達到 80%
        is_sst_all_sig = (
            (sum(p < 0.05 for p in p_sst_folds) / len(p_sst_folds) >= SIG_THRESHOLD)
            if len(p_sst_folds) > 0
            else False
        )

        r_sst_mean.append(m_sst)
        r_sst_err.append(s_sst_std)
        p_sst_sig.append(is_sst_all_sig)

        # CHL 統計量 (Mean, SD, 80% 門檻檢定)
        m_chl = np.mean(r_chl_folds) if len(r_chl_folds) > 0 else 0
        s_chl_std = np.std(r_chl_folds) if len(r_chl_folds) > 0 else 0
        # 判定 p < 0.05 的折數比例是否達到 80%
        is_chl_all_sig = (
            (sum(p < 0.05 for p in p_chl_folds) / len(p_chl_folds) >= SIG_THRESHOLD)
            if len(p_chl_folds) > 0
            else False
        )

        r_chl_mean.append(m_chl)
        r_chl_err.append(s_chl_std)
        p_chl_sig.append(is_chl_all_sig)

    # ==============================================================================
    # --- 5. 繪製圖表 ---
    # ==============================================================================
    fig, ax = plt.subplots(figsize=(12, 8))

    y_coords = np.arange(len(pc_list))
    bar_width = 0.35

    # 繪製 SST (上方長條) 與 CHL (下方長條)
    bars_sst = ax.barh(
        y_coords - bar_width / 2,
        r_sst_mean,
        bar_width,
        xerr=r_sst_err,
        capsize=5,
        error_kw={'elinewidth': 1.5, 'ecolor': 'black'},
        label='SST',
        color='royalblue',
        alpha=0.85,
    )
    bars_chl = ax.barh(
        y_coords + bar_width / 2,
        r_chl_mean,
        bar_width,
        xerr=r_chl_err,
        capsize=5,
        error_kw={'elinewidth': 1.5, 'ecolor': 'black'},
        label='CHL',
        color='darkorange',
        alpha=0.85,
    )

    # 顯著性打星號標註 (達 80% 折數顯著才標註)
    for idx, (bar, sig, err) in enumerate(zip(bars_sst, p_sst_sig, r_sst_err)):
        if sig:
            x_pos = bar.get_width() + (err if bar.get_width() >= 0 else -err)
            offset = 0.04 if x_pos >= 0 else -0.07
            ax.text(
                x_pos + offset,
                bar.get_y() + bar.get_height() / 2,
                '*',
                va='center',
                ha='center',
                fontsize=24,
                color='darkblue',
            )

    for idx, (bar, sig, err) in enumerate(zip(bars_chl, p_chl_sig, r_chl_err)):
        if sig:
            x_pos = bar.get_width() + (err if bar.get_width() >= 0 else -err)
            offset = 0.04 if x_pos >= 0 else -0.07
            ax.text(
                x_pos + offset,
                bar.get_y() + bar.get_height() / 2,
                '*',
                va='center',
                ha='center',
                fontsize=24,
                color='brown',
            )

    # 各變數間的分隔水平線
    for y in range(len(pc_list) - 1):
        ax.axhline(y + 0.5, color='black', linewidth=0.8, alpha=0.3)

    ax.set_yticks(y_coords)
    ax.set_yticklabels(pc_list, fontsize=20)
    ax.invert_yaxis()  # 依序由上而下：PO_PC1 -> AO_PC3

    # 軸線與標題設定
    ax.set_title(tag, loc='left', fontsize=26, pad=18)
    ax.set_xlabel('Correlation Coefficient (r)', fontsize=22)
    ax.tick_params(axis='x', labelsize=18)
    ax.axvline(0, color='black', linewidth=1.5)
    ax.set_xlim(-1.1, 1.1)
    ax.legend(loc='upper right', fontsize=16, frameon=True)

    plt.tight_layout()

    # 輸出存檔 (PNG 與 SVG)
    save_path_png = os.path.join(output_dir, f'Fig3_{tag}_65%.png')
    save_path_svg = os.path.join(output_dir, f'Fig3_{tag}_65%.svg')
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(save_path_svg, dpi=300, bbox_inches='tight')
    plt.close()

    print(f'已成功儲存圖表：{save_path_png}')

print('\n全部 4 張遮罩圖表已順利產生完成！')